# LAB 01｜第一次真实模型 API 调用

这份 Notebook 属于《AI 智能体大师课》第 2 课。你会先用 Mock 模式看懂完整数据流，再切换到你**有权使用**的 OpenAI Chat Completions 风格接口。

**安全底线**

- 不要把 API Key 写进代码单元、输出、截图、GitHub 或分享链接。
- 在 Colab 左侧的“钥匙”面板新建 `MODEL_API_KEY`，只授权本 Notebook 读取。
- 如果密钥曾出现在共享文档或仓库中，请先让接口管理员撤销或轮换。
- 本 Notebook 不内置任何供应商域名、模型名或密钥。


## 0. 学习目标与验收

运行结束时，你应当能解释：

1. Base URL、Endpoint、Header、JSON Body、Status Code 和 Response 的作用；
2. 为什么 `200` 只代表协议层成功，不能证明答案正确；
3. 为什么 GitHub Pages 不应保存长期 API Key；
4. 如何根据 `401 / 404 / 429 / 5xx` 定位不同故障层。

通关证据是一份**脱敏运行卡**，而不是密钥或完整请求 Header。

In [ ]:
#@title 1. 实验配置（先保持 Mock 模式）
DEMO_MODE = True #@param {type:"boolean"}
BASE_URL = "" #@param {type:"string"}
MODEL = "" #@param {type:"string"}
SECRET_NAME = "MODEL_API_KEY" #@param {type:"string"}
TIMEOUT_SECONDS = 60 #@param {type:"integer"}
MAX_RETRIES = 2 #@param {type:"integer"}

print("模式：", "Mock（不访问外部服务）" if DEMO_MODE else "真实 API")
print("Base URL：", "已填写" if BASE_URL.strip() else "未填写")
print("模型：", MODEL or "未填写")
print("Secret 名称：", SECRET_NAME)


### 切换真实接口前

1. 向接口管理员确认你有调用权限、服务条款、限流和当前模型 ID。
2. 把 `BASE_URL` 填成网关根地址、`MODEL` 填成可调用的模型 ID。
3. 在 Colab 左侧“钥匙”面板创建 `MODEL_API_KEY`，并打开本 Notebook 的访问开关。
4. 最后才把 `DEMO_MODE` 改为 `False`。

不要使用附件或他人发来的明文旧密钥；暴露过的凭证应先轮换。

In [ ]:
import json
import random
import socket
import time
from datetime import datetime, timezone
from getpass import getpass
from urllib.error import HTTPError, URLError
from urllib.parse import urlparse
from urllib.request import Request, urlopen


def normalize_chat_endpoint(base_url: str) -> str:
    """接受根地址、以 /v1 结尾的地址或完整 chat/completions 地址。"""
    value = base_url.strip().rstrip("/")
    if not value:
        raise ValueError("BASE_URL 为空。请先填写你有权使用的网关地址。")
    parsed = urlparse(value)
    if parsed.scheme != "https" or not parsed.hostname:
        raise ValueError("真实密钥只应发送到管理员确认过的 HTTPS 地址。")
    if parsed.username or parsed.password:
        raise ValueError("不要把用户名或密钥放进 URL。")
    if value.endswith("/chat/completions"):
        return value
    if value.endswith("/v1"):
        return value + "/chat/completions"
    return value + "/v1/chat/completions"


def read_secret(name: str) -> str:
    """优先读取 Colab Secret；本地运行时用隐藏输入，不打印返回值。"""
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except ImportError:
        value = getpass(f"请输入 {name}（输入不会显示）：")
    except Exception as exc:
        raise RuntimeError(
            f"无法读取 Colab Secret {name}。请在左侧钥匙面板创建并授权。"
        ) from exc
    if not value:
        raise RuntimeError(f"Secret {name} 为空。")
    return value


def extract_content(data: dict) -> str:
    """解析常见 Chat Completions 响应；不兼容时明确失败。"""
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        top_keys = sorted(data.keys()) if isinstance(data, dict) else []
        raise ValueError(
            "响应不是常见 Chat Completions 结构。"
            f"顶层字段为：{top_keys}。请查看网关文档并编写 adapter。"
        ) from exc
    if not isinstance(content, str) or not content.strip():
        raise ValueError("响应中的 message.content 不是非空字符串。")
    return content.strip()


print("基础函数已加载。真实模式下不会打印 Secret。")


In [ ]:
MOCK_CONTENT = """制定三日行程前必须确认：
1. 出发地、目的地与准确日期；
2. 总预算及交通、住宿、餐饮的预算边界；
3. 同行人数、年龄、行动能力与健康限制；
4. 兴趣优先级、旅行节奏与必须/不想去的地点；
5. 已订交通住宿、证件、天气风险及需要审批的动作。"""


def call_chat(messages: list[dict], *, temperature: float = 0.2, max_tokens: int = 500) -> dict:
    """返回只含脱敏运行信息的统一结果。"""
    started = time.perf_counter()
    timestamp = datetime.now(timezone.utc).isoformat()

    if DEMO_MODE:
        time.sleep(0.15)
        return {
            "mode": "mock",
            "status_code": 200,
            "latency_seconds": round(time.perf_counter() - started, 3),
            "request_id": "mock-local",
            "model": MODEL or "mock-travel-model",
            "content": MOCK_CONTENT,
            "timestamp_utc": timestamp,
        }

    if not MODEL.strip():
        raise ValueError("MODEL 为空。请填写管理员确认的模型 ID。")

    endpoint = normalize_chat_endpoint(BASE_URL)
    api_key = read_secret(SECRET_NAME)
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}",
    }
    payload = {
        "model": MODEL.strip(),
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    status_code = None
    response_text = ""
    response_headers = {}
    request = Request(
        endpoint,
        data=json.dumps(payload).encode("utf-8"),
        headers=headers,
        method="POST",
    )
    for attempt in range(MAX_RETRIES + 1):
        try:
            with urlopen(request, timeout=TIMEOUT_SECONDS) as response:
                status_code = response.status
                response_text = response.read().decode("utf-8", errors="replace")
                response_headers = {key.lower(): value for key, value in response.headers.items()}
        except HTTPError as exc:
            status_code = exc.code
            response_text = exc.read().decode("utf-8", errors="replace")
            response_headers = {key.lower(): value for key, value in exc.headers.items()}
        except (TimeoutError, socket.timeout) as exc:
            if attempt >= MAX_RETRIES:
                raise RuntimeError(f"请求超时，已尝试 {attempt + 1} 次。") from exc
            time.sleep(min(8, (2 ** attempt) + random.random()))
            continue
        except URLError as exc:
            raise RuntimeError(f"网络连接失败：{type(exc).__name__}。请检查域名、网络与 TLS。") from exc

        if status_code == 429 or 500 <= status_code < 600:
            if attempt < MAX_RETRIES:
                retry_after = response_headers.get("retry-after", "")
                try:
                    delay = float(retry_after)
                except ValueError:
                    delay = (2 ** attempt) + random.random()
                time.sleep(min(10, max(0.5, delay)))
                continue

        if not 200 <= status_code < 300:
            safe_preview = response_text[:300].replace(api_key, "[REDACTED]")
            raise RuntimeError(
                f"HTTP {status_code}。响应摘要：{safe_preview}"
            )
        break

    try:
        data = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise ValueError("服务返回了非 JSON 内容。请检查 Endpoint 或网关状态。") from exc

    content = extract_content(data)
    return {
        "mode": "real",
        "status_code": status_code,
        "latency_seconds": round(time.perf_counter() - started, 3),
        "request_id": response_headers.get("x-request-id", "not-provided"),
        "model": str(data.get("model") or MODEL),
        "content": content,
        "timestamp_utc": timestamp,
    }


print("通用客户端已就绪。重试只用于超时、429 和暂时性 5xx，且次数有限。")


In [ ]:
messages = [
    {
        "role": "system",
        "content": (
            "你是谨慎的旅行规划助手。当前只收集需求，不推荐具体地点，"
            "不编造用户没有提供的关键约束。"
        ),
    },
    {
        "role": "user",
        "content": "列出制定一份可执行三日旅行计划前必须确认的 5 项信息。",
    },
]

result = call_chat(messages, temperature=0.2, max_tokens=500)

print("运行模式：", result["mode"])
print("HTTP 状态：", result["status_code"])
print("耗时：", result["latency_seconds"], "秒")
print("模型：", result["model"])
print("请求 ID：", result["request_id"])
print("\n模型输出：\n" + result["content"])


In [ ]:
# 3. 最小验收：协议成功不等于业务成功，因此分层检查
checks = {
    "protocol_ok": 200 <= int(result["status_code"]) < 300,
    "content_is_text": isinstance(result["content"], str) and bool(result["content"].strip()),
    "mentions_five_items": all(str(i) in result["content"] for i in range(1, 6)),
    "receipt_has_no_secret_value": True,
}

if not DEMO_MODE:
    secret_value = read_secret(SECRET_NAME)
    public_receipt_text = json.dumps(
        {key: value for key, value in result.items() if key != "content"},
        ensure_ascii=False,
    )
    checks["receipt_has_no_secret_value"] = secret_value not in public_receipt_text

for name, passed in checks.items():
    print(("✅" if passed else "❌"), name)

assert checks["protocol_ok"], "协议层没有成功。先按 HTTP 状态码定位。"
assert checks["content_is_text"], "响应没有可用文本。"
assert checks["receipt_has_no_secret_value"], "停止分享：回执中检测到 Secret。"

print("\n注意：mentions_five_items 只是本课的简单启发式检查，不等于完整质量评测。")


## 4. 故障注入练习

真实模式下每次只改一个变量，并记录状态码与定位层：

| 故意制造的错误 | 预期现象 | 首先检查 |
|---|---|---|
| 把模型名改成不存在的值 | 常见为 400/404 | 模型 ID 与网关模型目录 |
| 把 Endpoint 多拼一个 `/v1` | 常见为 404 | URL 规范化 |
| 暂时把 Secret 改成无效值 | 常见为 401/403 | 鉴权与权限 |
| 快速并发发送过多请求 | 可能为 429 | 配额、速率与并发限制 |

不要为了练习去攻击或压测共享服务。并发/429 练习应使用管理员允许的低流量或后续提供的本地 Mock Server。

In [ ]:
# 5. 生成可分享的脱敏运行卡（不含 Prompt、模型全文、Base URL 或 Secret）
receipt = {
    "schema_version": 1,
    "lab": "LAB-01-first-api-call",
    "timestamp_utc": result["timestamp_utc"],
    "mode": result["mode"],
    "service": "institutional-gateway" if not DEMO_MODE else "local-mock",
    "model_alias": result["model"],
    "status_code": result["status_code"],
    "latency_seconds": result["latency_seconds"],
    "checks": checks,
}

print(json.dumps(receipt, ensure_ascii=False, indent=2))
print("\n提交前再次确认：上面没有 Base URL、API Key、Authorization Header 或私人旅行数据。")


## 通关任务

把上一个单元生成的脱敏运行卡，以及模型列出的 5 项待确认信息发给老师。**不要发送 API Key、完整 Header、私有 Base URL 或包含凭证的截图。**

## 一手参考

- [Google Colab FAQ](https://research.google.com/colaboratory/faq.html)
- [Google Colab Secrets 官方示例](https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/Secrets.ipynb)
- [IETF RFC 9110：HTTP Semantics](https://datatracker.ietf.org/doc/html/rfc9110)
- [IETF RFC 8259：JSON](https://datatracker.ietf.org/doc/html/rfc8259)
- [OpenAI Chat Completions API 形状参考](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)
- [vLLM OpenAI-Compatible Server](https://docs.vllm.ai/en/stable/serving/online_serving/openai_compatible_server/)
- [OWASP Secrets Management Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Secrets_Management_Cheat_Sheet.html)
